# 06 · Fase 2: efectos entre categorías

Estima, para cada categoría objetivo $j$, el modelo PPML de la **Fase 2** de `metodologia.tex`: la ecuación de la fase 1 más la intensidad promocional de las categorías del núcleo $k \neq j$ (día actual y dos bloques de rezagos) y la variable de control `p__RESTO`.

| | |
|---|---|
| **Entrada** | las de `05` más `exposicion_t44_categorias_dia.csv` y `exposicion_t44_resto_por_objetivo.csv` |
| **Salida** | `resultados/tablas/fase2_<CATEGORIA>.csv` y `resultados/tablas/fase2_cruzados.csv` (efectos $L_{jk}$ con corrección de Benjamini-Hochberg y clasificación) |
| **Especificación** | Igual que la fase 1: PPML, efectos fijos SKU, día de la semana y mes; errores agrupados por SKU y semana |

Los efectos cruzados varían solo por día (544 días como máximo), por eso el agrupamiento por semana es clave. El coeficiente de `p__RESTO` es un control y **no se interpreta**.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima el modelo PPML de la fase 2 (efectos entre categorías) por categoría objetivo.
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import re

import numpy as np
import pandas as pd
import pyfixest as pf
from statsmodels.stats.multitest import multipletests

## 1. Parámetros

In [2]:
# Categorías objetivo (las mismas de la fase 1)
# ------------------------------------------------------------------------------------------------------------
categorias_objetivo = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]

# D-28: núcleo de categorías de exposición (entran siempre como regresores, aunque no sean objetivo)
nucleo_exposicion = [
    "MEATS", "POULTRY", "DELI", "PRODUCE", "DAIRY",
    "BREAD/BAKERY", "BEVERAGES", "LIQUOR,WINE,BEER", "GROCERY I", "CLEANING",
]

categorias_con_programa = ["MEATS", "POULTRY", "SEAFOOD", "EGGS", "DELI", "PRODUCE"]
sufijos = ["", "_l1_3", "_l4_6"]
indicadores_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]
efectos_fijos = ["item_nbr", "dow", "mes"]
agrupar_por = ["item_nbr", "semana"]
dias_maximos = 544
tasa_fdr = 0.05                     # D-15: tasa de falsos descubrimientos de Benjamini-Hochberg

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
os.makedirs(ruta_tablas, exist_ok=True)
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_exposiciones = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")
archivo_categorias = os.path.join(ruta_procesados, "exposicion_t44_categorias_dia.csv")
archivo_resto = os.path.join(ruta_procesados, "exposicion_t44_resto_por_objetivo.csv")

## 2. Base de estimación de una categoría

Los pasos 1 a 3 son los de la fase 1. Luego se unen las exposiciones a otras categorías por fecha (solo el núcleo, **sin la propia categoría $j$**) y el resto por fecha y categoría objetivo.

In [3]:
def nombre_columna(familia: str) -> str:
    """
    Esta función convierte el nombre de una familia en el nombre de su columna de exposición (igual que en 03).

    :param familia:     el nombre de la familia (p. ej., 'LIQUOR,WINE,BEER').
    :return:            el nombre de columna (p. ej., 'p__LIQUOR_WINE_BEER').
    """

    return "p__" + re.sub(r"[^A-Z0-9]+", "_", familia.upper()).strip("_")


def columnas_cruzadas(categoria: str) -> list:
    """
    Esta función devuelve las columnas de exposición a otras categorías de una categoría objetivo: el núcleo sin la
    propia categoría y el resto, cada uno con sus bloques de rezagos.

    :param categoria:   el nombre de la categoría objetivo.
    :return:            la lista de nombres de columna.
    """

    fuentes = [nombre_columna(k) for k in nucleo_exposicion if k != categoria] + ["p__RESTO"]
    return [f"{c}{s}" for c in fuentes for s in sufijos]


def variables_propias() -> list:
    """
    Esta función devuelve los regresores de la fase 1 (promoción propia, de la clase y del resto de la categoría).

    :return:    la lista de nombres de columna.
    """

    return [f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos]


def construir_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de la fase 2 para una categoría objetivo.

    :param categoria:   el nombre de la categoría objetivo (p. ej., 'BEVERAGES').
    :return:            una tupla (base, exclusiones): la base SKU-día lista para estimar y un diccionario con el
                        número de filas en cada paso.
    """

    exclusiones = {}

    # 1. Filas del panel de la categoría
    panel = pd.read_parquet(
        archivo_panel,
        columns=["date", "item_nbr", "family", "unit_sales_ppml", "onpromotion"],
        filters=[("family", "==", categoria)],
    )
    panel["family"] = panel["family"].astype(str)
    assert len(panel) > 0, f"no hay filas para la categoría {categoria}"
    assert not panel.duplicated(["item_nbr", "date"]).any(), "el panel tiene más de una fila por SKU-día"
    n0 = len(panel)
    exclusiones["panel de la categoría"] = n0

    # 2. Exposiciones SKU-día (uno a uno)
    columnas_expo = ["item_nbr", "date", "dia_programa", "promo_clase_otros", "promo_cat_otras_clases"] + [
        f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos[1:]
    ]
    expo = pd.read_parquet(archivo_exposiciones, columns=columnas_expo)
    base = panel.merge(expo, on=["item_nbr", "date"], how="left", validate="one_to_one")
    assert len(base) == n0, "la unión con exposiciones cambió el número de filas"
    assert base["dia_programa"].notna().all(), "hay SKU-días del panel sin exposiciones"

    # 3. Calendario (muchos a uno por fecha)
    calendario = pd.read_csv(archivo_calendario, parse_dates=["date"])
    base = base.merge(
        calendario[["date", "dow", "mes", "semana", "ventana_terremoto"] + indicadores_calendario],
        on="date", how="left", validate="many_to_one",
    )
    assert len(base) == n0, "la unión con el calendario cambió el número de filas"
    assert base["dow"].notna().all(), "hay fechas del panel sin calendario"

    # 4. Exposición a las otras categorías del núcleo (por fecha), sin la propia categoría
    cols_nucleo = [f"{nombre_columna(k)}{s}" for k in nucleo_exposicion if k != categoria for s in sufijos]
    cat_dia = pd.read_csv(archivo_categorias, usecols=["date"] + cols_nucleo, parse_dates=["date"])
    assert not cat_dia["date"].duplicated().any(), "exposicion_t44_categorias_dia debe tener una fila por día"
    base = base.merge(cat_dia, on="date", how="left", validate="many_to_one")
    assert len(base) == n0, "la unión con las categorías cambió el número de filas"
    assert nombre_columna(categoria) not in base.columns or categoria not in nucleo_exposicion, \
        "la columna de la propia categoría no debe estar entre los regresores"

    # 5. Resto por categoría objetivo (por fecha y categoría)
    cols_resto = [f"p__RESTO{s}" for s in sufijos]
    resto = pd.read_csv(archivo_resto, usecols=["family", "date"] + cols_resto, parse_dates=["date"])
    resto = resto.loc[resto["family"] == categoria].drop(columns="family")
    assert not resto["date"].duplicated().any(), "el resto debe tener una fila por día para la categoría"
    base = base.merge(resto, on="date", how="left", validate="many_to_one")
    assert len(base) == n0, "la unión con el resto cambió el número de filas"
    assert not base.duplicated(["item_nbr", "date"]).any(), "la base tiene más de una fila por SKU-día"

    # 6. Exclusiones: terremoto, días de programa (si corresponde) y nulos
    base = base.loc[~base["ventana_terremoto"]]
    exclusiones["tras excluir ventana del terremoto"] = len(base)
    if categoria in categorias_con_programa:
        base = base.loc[~base["dia_programa"]]
    exclusiones["tras excluir días de programa"] = len(base)
    base["onpromotion"] = base["onpromotion"].astype(float)
    base = base.dropna(subset=variables_propias() + columnas_cruzadas(categoria) + ["unit_sales_ppml"])
    exclusiones["tras excluir nulos (base final)"] = len(base)

    assert not base.duplicated(["item_nbr", "date"]).any(), "la base final tiene más de una fila por SKU-día"
    return base.reset_index(drop=True), exclusiones

## 3. Estimación

Se guardan todos los coeficientes de promoción (propios, de clase, de categoría, cruzados y resto). Para los efectos cruzados contemporáneos se calcula $L_{jk}=e^{0{,}1\,\beta_{jk,0}}-1$ con su intervalo de confianza.

In [4]:
def estimar_fase2(base: pd.DataFrame, categoria: str) -> pd.DataFrame:
    """
    Esta función estima el PPML de la fase 2 y devuelve la tabla de coeficientes de promoción.

    :param base:        la base de estimación de la categoría (salida de construir_base).
    :param categoria:   el nombre de la categoría objetivo.
    :return:            un DataFrame con una fila por coeficiente de promoción: coeficiente, error estándar, valor p,
                        efecto en porcentaje, tipo de variable, categoría fuente y tamaño de la muestra.
    """

    # Conservar solo los indicadores de calendario con variación
    calendario_activo = [c for c in indicadores_calendario if base[c].nunique() > 1]
    base = base.assign(**{c: base[c].astype(float) for c in calendario_activo})

    # Estimar
    regresores = variables_propias() + columnas_cruzadas(categoria) + calendario_activo
    formula = f"unit_sales_ppml ~ {' + '.join(regresores)} | {' + '.join(efectos_fijos)}"
    modelo = pf.fepois(formula, data=base, vcov={"CRV1": "+".join(agrupar_por)})

    tabla = modelo.tidy().reset_index().rename(columns={"Coefficient": "variable"})
    tabla = tabla.rename(columns={"Estimate": "coef", "Std. Error": "se", "Pr(>|t|)": "p_valor"})
    tabla = tabla[["variable", "coef", "se", "p_valor"]]
    tabla = tabla.loc[tabla["variable"].isin(variables_propias() + columnas_cruzadas(categoria))].copy()

    # Identificar la categoría fuente y el bloque de cada variable
    mapa = {nombre_columna(k): k for k in nucleo_exposicion}
    mapa["p__RESTO"] = "RESTO (control)"
    def fuente(v):
        for s in ["_l1_3", "_l4_6"]:
            if v.endswith(s):
                return v[: -len(s)], s.strip("_")
        return v, "dia"
    tabla["base_var"] = tabla["variable"].map(lambda v: fuente(v)[0])
    tabla["bloque"] = tabla["variable"].map(lambda v: fuente(v)[1])
    tabla["fuente"] = tabla["base_var"].map(mapa).fillna("propia/clase/categoria")

    # Efecto en porcentaje del coeficiente y, para los cruzados, de +10 puntos porcentuales: L = exp(0,1 beta) - 1
    tabla["efecto_pct"] = 100 * (np.exp(tabla["coef"]) - 1)
    tabla["L_10pp_pct"] = 100 * (np.exp(0.1 * tabla["coef"]) - 1)
    tabla["L_ic95_inf_pct"] = 100 * (np.exp(0.1 * (tabla["coef"] - 1.96 * tabla["se"])) - 1)
    tabla["L_ic95_sup_pct"] = 100 * (np.exp(0.1 * (tabla["coef"] + 1.96 * tabla["se"])) - 1)

    tabla.insert(0, "categoria", categoria)
    tabla["n_obs"] = len(base)
    tabla["n_skus"] = base["item_nbr"].nunique()
    tabla["n_dias"] = base["date"].nunique()
    return tabla.reset_index(drop=True)

In [5]:
# Estimar cada categoría objetivo y guardar su tabla
for categoria in categorias_objetivo:
    print(f"\n=== {categoria} ===")
    base, exclusiones = construir_base(categoria)
    for paso, n in exclusiones.items():
        print(f"  {paso:<40} {n:>10,}")
    dias_usados = base["date"].nunique()
    assert dias_usados <= dias_maximos, f"se usan {dias_usados} días y el máximo es {dias_maximos}: falta una exclusión"
    print(f"  SKUs: {base['item_nbr'].nunique():,} | días usados: {dias_usados} (máximo {dias_maximos})")

    tabla = estimar_fase2(base, categoria)
    nombre_archivo = re.sub(r"[^A-Za-z0-9]+", "_", categoria).strip("_")
    tabla.to_csv(os.path.join(ruta_tablas, f"fase2_{nombre_archivo}.csv"), index=False)

    # Controles: efecto propio positivo y coeficientes razonables
    propio = tabla.loc[tabla["variable"] == "onpromotion"].iloc[0]
    print(f"  Efecto propio del día: {propio['efecto_pct']:+.1f}% (p = {propio['p_valor']:.4f})")
    if propio["coef"] <= 0:
        print("  >>> ATENCIÓN: el efecto propio del día es <= 0; revisar antes de interpretar")
    grandes = tabla.loc[tabla["coef"].abs() > 5, "variable"].tolist()
    if grandes:
        print(f"  >>> ATENCIÓN: coeficientes con |valor| > 5 (posible colinealidad o separación): {grandes}")


=== BEVERAGES ===
  panel de la categoría                       287,407
  tras excluir ventana del terremoto          266,744
  tras excluir días de programa               266,744
  tras excluir nulos (base final)             258,915
  SKUs: 576 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


  Efecto propio del día: +59.0% (p = 0.0000)

=== GROCERY I ===
  panel de la categoría                       620,919
  tras excluir ventana del terremoto          574,218
  tras excluir días de programa               574,218
  tras excluir nulos (base final)             556,533
  SKUs: 1,153 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


  Efecto propio del día: +60.4% (p = 0.0000)

=== CLEANING ===
  panel de la categoría                       220,098
  tras excluir ventana del terremoto          203,350
  tras excluir días de programa               203,350
  tras excluir nulos (base final)             197,454
  SKUs: 396 | días usados: 532 (máximo 544)
  Efecto propio del día: +88.6% (p = 0.0000)

=== DAIRY ===
  panel de la categoría                       122,674
  tras excluir ventana del terremoto          113,450
  tras excluir días de programa               113,450
  tras excluir nulos (base final)             109,779
  SKUs: 226 | días usados: 532 (máximo 544)
  Efecto propio del día: +41.2% (p = 0.0000)

=== BREAD/BAKERY ===
  panel de la categoría                        60,669
  tras excluir ventana del terremoto           56,026
  tras excluir días de programa                56,026
  tras excluir nulos (base final)              54,466
  SKUs: 105 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


  Efecto propio del día: +45.3% (p = 0.0000)

=== LIQUOR,WINE,BEER ===
  panel de la categoría                        27,102
  tras excluir ventana del terremoto           25,148
  tras excluir días de programa                25,148
  tras excluir nulos (base final)              21,466
  SKUs: 49 | días usados: 532 (máximo 544)
  Efecto propio del día: +75.0% (p = 0.0000)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


## 4. Efectos cruzados contemporáneos y corrección de Benjamini-Hochberg

Se toman todos los efectos cruzados del día actual (categorías del núcleo $k\neq j$; **sin** `p__RESTO`) de todas las categorías estimadas, se corrigen juntos por Benjamini-Hochberg (FDR 5%) y se clasifica cada par.

In [6]:
# Reunir las tablas de la fase 2 que existan en la carpeta
archivos = sorted(f for f in os.listdir(ruta_tablas) if f.startswith("fase2_") and f != "fase2_cruzados.csv")
todas = pd.concat([pd.read_csv(os.path.join(ruta_tablas, f)) for f in archivos], ignore_index=True)

# Efectos cruzados contemporáneos entre categorías del núcleo
nombres_nucleo = {nombre_columna(k): k for k in nucleo_exposicion}
cruzados = todas.loc[(todas["bloque"] == "dia") & todas["base_var"].isin(nombres_nucleo)].copy()
cruzados = cruzados.rename(columns={"categoria": "afectada_j", "fuente": "en_promocion_k"})

# Corrección de Benjamini-Hochberg sobre todos los pares
rechaza, p_ajustado, _, _ = multipletests(cruzados["p_valor"], alpha=tasa_fdr, method="fdr_bh")
cruzados["p_bh"] = p_ajustado
cruzados["significativo_bh"] = rechaza
cruzados["clasificacion"] = np.where(
    ~rechaza, "sin efecto", np.where(cruzados["coef"] > 0, "sinergia", "canibalizacion")
)

columnas = ["afectada_j", "en_promocion_k", "coef", "se", "p_valor", "p_bh", "L_10pp_pct", "L_ic95_inf_pct",
            "L_ic95_sup_pct", "significativo_bh", "clasificacion", "n_obs", "n_dias"]
cruzados = cruzados[columnas].sort_values(["afectada_j", "en_promocion_k"]).reset_index(drop=True)
cruzados.to_csv(os.path.join(ruta_tablas, "fase2_cruzados.csv"), index=False)

# Verificar el número de pares: (categorías objetivo) x (categorías del núcleo distintas de la propia)
esperados = sum(len([k for k in nucleo_exposicion if k != j]) for j in todas["categoria"].unique())
assert len(cruzados) == esperados, f"se esperaban {esperados} pares y hay {len(cruzados)}"

print(f"Pares estimados: {len(cruzados)} (corrección BH, FDR {tasa_fdr:.0%})")
print(cruzados["clasificacion"].value_counts().to_string())
print()
print(cruzados.loc[cruzados["significativo_bh"],
                   ["afectada_j", "en_promocion_k", "L_10pp_pct", "p_bh", "clasificacion"]].round(4).to_string(index=False))

print(f"\n>>>> Proceso finalizado <<<<")

Pares estimados: 54 (corrección BH, FDR 5%)
clasificacion
sin efecto        51
sinergia           2
canibalizacion     1

      afectada_j en_promocion_k  L_10pp_pct   p_bh  clasificacion
LIQUOR,WINE,BEER      GROCERY I     25.8312 0.0341       sinergia
LIQUOR,WINE,BEER          MEATS      1.9997 0.0144       sinergia
LIQUOR,WINE,BEER        POULTRY    -10.8461 0.0341 canibalizacion

>>>> Proceso finalizado <<<<
